### Dyna-Q 算法

#### 简介
- Dyna-Q算法是基于模型的强化学习算法, 不过它的环境模型是通过采样数据估计得到的
- 强化学习算法两个重要的评价指标：一个是算法收敛后的策略在初始状态下的期望回报，另一个是样本复杂度，即算法达到收敛结果需要在真实环境中采样的样本数量


#### Dyna-Q
- 定义: 通过真实环境交互学习环境模型，并结合真实经验与模型生成的模拟经验，使用 Q-learning 更新动作价值函数
- 在每次与环境进行交互执行一次 Q-learning 之后, Dyna-Q 会做 $n$ 次 Q-planning, 利用环境模型生成的模拟数据接着做更新

核心公式: 
- 环境模型:

$$
M(s,a) = (\hat{r}, \hat{s}')
$$

- Q-planning 更新:

$$
Q(s,a) \leftarrow Q(s,a) + \alpha \left[\hat{r} + \gamma \max_{a'} Q(\hat{s}',a') - Q(s,a)\right]
$$


In [ ]:
import numpy as np  # 导入NumPy科学计算库
import random       # 导入random库，用于随机选择历史经验

class DynaQ:
    """Dyna-Q算法"""
    def __init__(self, ncol, nrow, epsilon, alpha, gamma, n_planning, n_action=4):  # 初始化Dyna-Q智能体
        self.Q_table = np.zeros([nrow * ncol, n_action])  # 初始化状态数×动作数的全0 Q表
        self.n_action = n_action      # 保存动作数量
        self.alpha = alpha            # 设置学习率
        self.gamma = gamma            # 设置折扣因子
        self.epsilon = epsilon        # 设置ε-Greedy的随机探索概率

        self.n_planning = n_planning  # 设置每次真实交互后执行Q-planning的次数
        self.model = dict()           # 创建空字典，用于保存学到的环境模型

    def take_action(self, state):     # 使用ε-Greedy策略选择动作
        if np.random.random() < self.epsilon:          # 以ε的概率随机探索
            action = np.random.randint(self.n_action)  # 随机选择一个动作
        else:                                          # 以1-ε的概率利用当前Q值
            action = np.argmax(self.Q_table[state])    # 选择当前状态下Q值最大的动作
        return action                                  # 返回选中的动作编号

    def q_learning(self, s0, a0, r, s1):               # 使用一条状态转移经验执行Q-learning更新
        td_error = r + self.gamma * self.Q_table[s1].max() - self.Q_table[s0, a0]  # 计算TD误差
        self.Q_table[s0, a0] += self.alpha * td_error  # 根据TD误差更新当前Q值

    def update(self, s0, a0, r, s1):      # 使用真实经验和环境模型共同更新Q表
        self.q_learning(s0, a0, r, s1)    # 第一步：使用真实环境经验进行一次Q-learning更新
        self.model[(s0, a0)] = r, s1      # 第二步：保存状态动作对及对应的奖励和下一状态

        for _ in range(self.n_planning):  # 第三步：循环执行n_planning次模拟规划
            (s, a), (r, s_) = random.choice(list(self.model.items()))  # 从环境模型中随机抽取一条历史状态转移经验
            self.q_learning(s, a, r, s_)  # 使用模型提供的经验再次更新Q表
